In [ ]:
import sys
import os


env = os.environ.copy()
env["PYTHONPATH"] = f"/kaggle/working:{env.get('PYTHONPATH', '')}"

# bp = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script'

# sys.path = [p for p in sys.path if bp not in p]

# # 3. Specifically delete torchvision from memory if it was already ghost-loaded
# if "torchvision" in sys.modules:
#     del sys.modules["torchvision"]
# if "torch" in sys.modules:
#     del sys.modules["torch"]

# 1. Upgrade pip (using standard pip as uv handles itself)
# pip install --no-index --find-links=/kaggle/input/datasets/nesfan/sem-packages/offline_packages --upgrade pip

# 2. Install initial batch
!uv pip install --no-index --find-links=/kaggle/input/datasets/nesfan/sem-packages/offline_packages huggingface_hub accelerate scikit-learn

# 3. Uninstall specific libs
!uv pip uninstall torchvision pandas

# 4. Uninstall transformers stack to ensure clean versioning
!uv pip uninstall transformers tokenizers accelerate

# 5. Install specific versions
!uv pip install --no-index --find-links=/kaggle/input/datasets/nesfan/sem-packages/offline_packages "transformers==4.56.0" "protobuf==5.29.3"

# 6. Install core ML libs
!uv pip install --force-reinstall --no-index --find-links=/kaggle/input/datasets/nesfan/sem-packages/offline_packages torch datasets

# 7. Install utility libs
!uv pip install --no-index --find-links=/kaggle/input/datasets/nesfan/sem-packages/offline_packages tqdm wandb

# 8. Install optimization and transfer libs
!uv pip install --no-index --find-links=/kaggle/input/datasets/nesfan/sem-packages/offline_packages bitsandbytes accelerate hf_transfer

# 9. Force reinstall numpy 2.0
!uv pip install --no-index --find-links=/kaggle/input/datasets/nesfan/sem-packages/offline_packages --force-reinstall --no-cache-dir "numpy==2.0" huggingface_hub==0.36.2

In [ ]:
import os

# !rm -r /kaggle/working/graphcodebert_flat /kaggle/working/.cache/huggingface/hub
# os.environ["HF_HOME"] = "/kaggle/working/.cache/huggingface"
# os.environ["HF_HUB_OFFLINE"] = "0"
# os.environ["TRANSFORMERS_OFFLINE"] = "0"

# from huggingface_hub import snapshot_download
# snapshot_download(
#     repo_id="microsoft/graphcodebert-base",
#     local_dir="/kaggle/working/graphcodebert_flat",
#     local_dir_use_symlinks=False,
# )""
# os.environ["HF_HOME"] = "/kaggle/working/.cache/huggingface"
# os.environ["HF_HUB_OFFLINE"] = "1"
# os.environ["TRANSFORMERS_OFFLINE"] = "1"

# from transformers import (
#     AutoTokenizer,
#     AutoConfig,
#     AutoModelForSequenceClassification,
#     DataCollatorWithPadding,
# )

# model_name="/kaggle/working/graphcodebert_flat"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# config = AutoConfig.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(
#     model_name,
#     config=config,
#     ignore_mismatched_sizes=True
# )

!ls /kaggle/working/.cache/huggingface/
# !echo $HF_HUB_CACHE

In [ ]:
# Install PyTorch/XLA for TPU (Kaggle)
# !uv pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

import os
os.environ["HF_HOME"] = "/kaggle/working/.cache/huggingface"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"

import re
import logging
import sys
import subprocess
from pathlib import Path
import json
import shutil

import torch
import torch.nn as nn
# import torch_xla
# import torch_xla.core.xla_model as xm

from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from IPython import get_ipython
import wandb
from typing import Dict, Tuple, Callable, Optional
from dataclasses import dataclass, field, asdict
import numpy as np
import random
import yaml
import torch.nn.functional as F
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

# ----------------------------------------------------------------------
# Environment & helpers
# ----------------------------------------------------------------------
def configure_environment_paths():
    try:
        if "google.colab" in str(get_ipython()):
            print("✅ Environment: Google Colab")
            base_data_path = "/content/"
            base_output_path = "/content/"
            environment_name = "colab"
        elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            print("✅ Environment: Kaggle")
            base_data_path = "/kaggle/input/"
            base_output_path = "/kaggle/working/"
            environment_name = "kaggle"
        else:
            print("⚠️ Environment: Local/Unknown")
            base_data_path = "./data/"
            base_output_path = "./output/"
            environment_name = "local"
    except NameError:
        print("⚠️ Non-interactive session. Using local paths.")
        base_data_path = "./data/"
        base_output_path = "./output/"
        environment_name = "local"
    os.makedirs(base_output_path, exist_ok=True)
    print(f"📂 Data Path: {base_data_path}")
    print(f"📦 Output Path: {base_output_path}")
    return base_data_path, base_output_path, environment_name

def load_secret(key_name: str, env_name: str) -> str | None:
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env_name}' environment...")
    try:
        if env_name == "colab":
            from google.colab import userdata
            secret_value = userdata.get(key_name)
        elif env_name == "kaggle":
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"⚠️ Secret '{key_name}' not found in the {env_name} environment.")
            return None
        print(f"✅ Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"❌ An error occurred while loading secret '{key_name}': {e}")
        return None

def print_system_info():
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch
        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
        # TPU detection
        try:
            import torch_xla
            print(f"PyTorch/XLA version: {torch_xla.__version__}")
            print(f"TPU cores: {xm.xrt_world_size()}")
        except:
            pass
    except ImportError:
        print("PyTorch not installed")

def configure_wandb_offline(output_dir: str, project: str = "machine-generated-code-detection") -> str:
    wandb_dir = os.path.join(output_dir, "wandb")
    os.makedirs(wandb_dir, exist_ok=True)
    os.environ["WANDB_MODE"] = "offline"
    os.environ["WANDB_DIR"] = wandb_dir
    os.environ["WANDB_PROJECT"] = project
    return wandb_dir

INPUT_PATH, OUTPUT_PATH, ENV_NAME = configure_environment_paths()
is_kaggle = "kaggle" in ENV_NAME
is_colab = not is_kaggle
print_system_info()

os.environ["HF_TOKEN"] = "thisismyhftoken"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ----------------------------------------------------------------------
# Logging
# ----------------------------------------------------------------------
def setup_logger(output_dir: str, name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(console_handler)
    os.makedirs(output_dir, exist_ok=True)
    log_path = Path(output_dir) / f"{name}.log"
    file_handler = logging.FileHandler(log_path, mode="w")
    file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(file_handler)
    return logger
logger = setup_logger(".", "notebook")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

# ----------------------------------------------------------------------
# Data preprocessing & tokenization (with caching)
# ----------------------------------------------------------------------
def preprocess_code(code_str: str) -> str:
    code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
    code_str = re.sub(r"\r\n|\r", "\n", code_str)
    if random.random() < 0.2:
        code_str = code_str.replace("    ", "\t")
    elif random.random() < 0.2:
        code_str = code_str.replace("    ", "  ")
    code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
    code_str = re.sub(r"\n{3,}", "\n\n", code_str)
    code_str = re.sub(r"[ \t]+", " ", code_str)
    return code_str.strip()

def tokenize_function(examples: Dict, tokenizer, max_length: int = 512) -> Dict:
    codes = [preprocess_code(c) for c in examples["code"]]
    return tokenizer(codes, truncation=True, max_length=max_length, padding=False)

def display_dataset_info(dataset: Dataset, name: str, num_samples: int = 3):
    print(f"\n{'='*50}")
    print(f"📊 Dataset: {name}")
    print(f"{'='*50}")
    print(f"Number of examples: {len(dataset)}")
    print(f"Column names: {dataset.column_names}")
    print("\nColumn types:")
    for col in dataset.column_names:
        dtype = dataset.features[col].dtype if hasattr(dataset.features[col], 'dtype') else type(dataset[0][col])
        print(f"  {col}: {dtype}")
    print(f"\n🔍 First {num_samples} samples:")
    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        print(f"\n--- Sample {i+1} ---")
        for key, value in sample.items():
            if isinstance(value, str) and len(value) > 200:
                value = value[:200] + "..."
            print(f"  {key}: {value}")
    print(f"{'='*50}\n")

def get_tokenized_cache_path(cache_dir: str, split: str, max_length: int, limit: int = None) -> Path:
    limit_suffix = f"_limit{limit}" if limit else ""
    filename = f"{split}_maxlen{max_length}{limit_suffix}"
    return Path(cache_dir) / filename

def load_datasets(tokenizer, max_length: int = 512, aug=None, limit: int = None, cache_dir: str = "./tokenized_cache") -> Tuple[Dataset, Dataset]:
    os.makedirs(cache_dir, exist_ok=True)
    train_dataset = load_dataset("parquet", data_files="/kaggle/input/datasets/dzung271828/semeval-taska/train.parquet", split="train")
    val_dataset = load_dataset("parquet", data_files="/kaggle/input/datasets/dzung271828/semeval-taska/validation.parquet", split="train")

    display_dataset_info(train_dataset, "Train (raw)")
    display_dataset_info(val_dataset, "Validation (raw)")

    if limit is not None:
        logger.info(f"🚩 TEST MODE: Limiting datasets to first {limit} samples.")
        train_dataset = train_dataset.select(range(min(limit, len(train_dataset))))
        val_dataset = val_dataset.select(range(min(limit, len(val_dataset))))

    if aug is not None:
        logger.info("Applying data augmentation before tokenization...")
        def augment_batch(examples):
            examples["code"] = [aug(c) for c in examples["code"]]
            return examples
        train_dataset = train_dataset.map(augment_batch, batched=True, batch_size=512, desc="Augmenting train", num_proc=os.cpu_count())

    logger.info("Tokenizing datasets...")
    train_dataset = train_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True, batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in train_dataset.column_names],
        desc="Tokenizing train", num_proc=os.cpu_count()
    )
    val_dataset = val_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True, batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in val_dataset.column_names],
        desc="Tokenizing val", num_proc=os.cpu_count()
    )

    train_dataset = train_dataset.rename_column("label", "labels")
    val_dataset = val_dataset.rename_column("label", "labels")

    return train_dataset, val_dataset

def compute_metrics_fn(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "macro_f1": f1}

# ------------------------------
# FFT low-pass filter function
# ------------------------------
def low_pass_filter_embeddings(embeddings, keep_ratio=0.5):
    """
    Apply low‑pass filter to embeddings in the frequency domain.
    Args:
        embeddings: torch.Tensor of shape (batch, seq_len, hidden_dim)
        keep_ratio: fraction of low frequencies to keep (0.0 to 1.0)
    Returns:
        filtered_embeddings: same shape as input
    """
    batch, seq_len, hidden = embeddings.shape
    # FFT along sequence dimension (dim=1)
    fft = torch.fft.rfft(embeddings, dim=1, norm='ortho')
    keep_len = max(1, int(seq_len * keep_ratio))
    fft[:, keep_len:] = 0.0
    filtered = torch.fft.irfft(fft, n=seq_len, dim=1, norm='ortho')
    return filtered


# ----------------------------------------------------------------------
# Code augmentation (unchanged)
# ----------------------------------------------------------------------
class CodeAugmentation:
    def __init__(self, rename_prob=0.3, format_prob=0.3, mixup_alpha=1.0):
        self.rename_prob = rename_prob
        self.format_prob = format_prob
        self.mixup_alpha = mixup_alpha

    def __call__(self, text: str) -> str:
        text = str(text)
        if random.random() < self.rename_prob:
            import re
            tokens = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', text)
            keywords = {'def', 'return', 'if', 'else', 'for', 'while', 'import',
                        'from', 'as', 'with', 'try', 'except', 'class', 'lambda',
                        'and', 'or', 'not', 'True', 'False', 'None'}
            identifiers = [t for t in set(tokens) if t not in keywords and len(t) > 1]
            if identifiers:
                rename_map = {id_: f"var_{random.randint(1000,9999)}" for id_ in identifiers[:5]}
                for old, new in rename_map.items():
                    text = text.replace(old, new)
        if random.random() < self.format_prob:
            lines = text.split('\n')
            new_lines = []
            for line in lines:
                new_lines.append(line)
                if random.random() < 0.1:
                    new_lines.append('')
            text = '\n'.join(new_lines)
            if random.random() < 0.2:
                text = '\n'.join(' ' + line if random.random() < 0.1 else line for line in text.split('\n'))
        return text

# ----------------------------------------------------------------------
# Loss factories (unchanged)
# ----------------------------------------------------------------------
def get_label_smoothed_cross_entropy(smoothing: float) -> Callable:
    def loss_fn(outputs, labels, **_):
        logits = outputs.logits
        n_classes = logits.size(-1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits).fill_(smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
        return -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()
    return loss_fn

def get_focal_loss(alpha: float, gamma: float, smoothing: float = 0.0) -> Callable:
    def focal_loss(outputs, labels=None, soft_labels=None, **_):
        logits = outputs.logits
        if soft_labels is not None:
            probs = F.softmax(logits, dim=-1)
            pt = (soft_labels * probs).sum(dim=-1)
            ce = -(soft_labels * F.log_softmax(logits, dim=-1)).sum(dim=-1)
        elif smoothing > 0:
            n_classes = logits.size(-1)
            smooth_targets = torch.zeros_like(logits).fill_(smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
            ce = -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1)
            probs = F.softmax(logits, dim=-1)
            pt = (smooth_targets * probs).sum(dim=-1)
        else:
            ce = F.cross_entropy(logits, labels, reduction="none")
            pt = torch.exp(-ce)
        loss = alpha * (1 - pt) ** gamma * ce
        return loss.mean()
    return focal_loss

def get_infonce_loss(temperature: float, weight: float) -> Callable:
    def infonce_loss(outputs, labels, **_):
        reps = outputs.hidden_states[-1]
        reps = F.normalize(reps, dim=-1)
        sim_matrix = torch.mm(reps, reps.transpose(-1, -2)) / temperature
        target = torch.arange(reps.size(0), device=reps.device)
        loss = F.cross_entropy(sim_matrix, target)
        return weight * loss
    return infonce_loss

# ----------------------------------------------------------------------
# Model loading with attention outputs
# ----------------------------------------------------------------------
def load_model_and_tokenizer(cfg):
    logger.info(f"Loading tokenizer from: {cfg.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

    logger.info(f"Loading config from: {cfg.model_name}")
    config = AutoConfig.from_pretrained(cfg.model_name)
    config.num_labels = cfg.num_labels
    config.problem_type = "single_label_classification"
    config.hidden_dropout_prob = cfg.hidden_dropout_prob
    config.attention_probs_dropout_prob = cfg.attention_probs_dropout_prob
    if hasattr(config, 'classifier_dropout'):
        config.classifier_dropout = cfg.classifier_dropout

    if cfg.use_attn_spectral:
        config.output_attentions = True
        logger.info("Enabled attention output for spectral loss.")

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        config=config,
        ignore_mismatched_sizes=True
    )
    logger.info(f"Model loaded successfully. num_labels={model.config.num_labels}, hidden_size={model.config.hidden_size}")
    model = apply_spectral_norm(model, cfg)
    logger.info("Applied spectral normalization, actually done or not is in the function itself!")

    if cfg.gradient_checkpointing:
        model.gradient_checkpointing_enable()
        logger.info("Gradient checkpointing enabled")

    return model, tokenizer
    
def apply_spectral_norm(model, cfg):
    """Apply spectral normalisation to the classifier head."""
    if not cfg.use_spectral_norm:
        return model
    # Get classifier module (works for AutoModelForSequenceClassification)
    if hasattr(model, 'classifier'):
        model.classifier.dense = torch.nn.utils.parametrizations.spectral_norm(model.classifier.dense)
        logger.info("Applied spectral norm to classifier")
    elif hasattr(model, 'score'):
        model.score.dense = torch.nn.utils.parametrizations.spectral_norm(model.score.dense)
        logger.info("Applied spectral norm to score layer")
    else:
        logger.warning("Could not find classifier head for spectral norm")
    return model
    
def freeze_base_model(model):
    for name, param in model.named_parameters():
        if "classifier" not in name and "cls" not in name.lower():
            param.requires_grad = False
    logger.info("Base model frozen - only classifier head is trainable")

def log_training_config(cfg, logger):
    logger.info("===== Training Configuration =====")
    for field_name, field_value in cfg.__dataclass_fields__.items():
        value = getattr(cfg, field_name)
        if field_name == "device":
            continue
        logger.info(f"{field_name:20} : {value}")
    logger.info("=================================")

def log_model_architecture(model, tokenizer, logger):
    logger.info("===== Model Architecture =====")
    logger.info("\n" + model.__repr__())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params
    logger.info("===== Parameter Summary =====")
    logger.info(f"Total Parameters:         {total_params:,}")
    logger.info(f"Trainable Parameters:     {trainable_params:,}")
    logger.info(f"Non-trainable Parameters: {non_trainable_params:,}")
    logger.info("===== Tokenizer Summary =====")
    logger.info(f"Vocab size: {len(tokenizer)} | Special tokens: {tokenizer.all_special_tokens}")
    logger.info("===== End of Architecture Log =====")

# ----------------------------------------------------------------------
# Config logging callback
# ----------------------------------------------------------------------
class ConfigLoggingCallback(TrainerCallback):
    def __init__(self, train_config):
        self.train_config = train_config

    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        if not os.path.exists(checkpoint_dir):
            logger.warning(f"Expected checkpoint directory not found: {checkpoint_dir}")
            return
        config_dict = self._prepare_config_dict(args, state)
        config_path = os.path.join(checkpoint_dir, "config_hyperparams.json")
        try:
            with open(config_path, "w") as f:
                json.dump(config_dict, f, indent=2, default=str)
            logger.info(f"✅ Saved hyperparameters to {config_path}")
        except Exception as e:
            logger.error(f"❌ Failed to save config to {config_path}: {e}")

    def _prepare_config_dict(self, training_args, state):
        train_cfg_dict = {
            k: getattr(self.train_config, k) for k in self.train_config.__dataclass_fields__.keys()
        }
        training_args_dict = {
            "output_dir": training_args.output_dir,
            "num_train_epochs": training_args.num_train_epochs,
            "per_device_train_batch_size": training_args.per_device_train_batch_size,
            "per_device_eval_batch_size": training_args.per_device_eval_batch_size,
            "learning_rate": training_args.learning_rate,
            "warmup_steps": training_args.warmup_steps,
            "weight_decay": training_args.weight_decay,
            "logging_steps": training_args.logging_steps,
            "eval_steps": training_args.eval_steps,
            "save_steps": training_args.save_steps,
            "metric_for_best_model": training_args.metric_for_best_model,
            "greater_is_better": training_args.greater_is_better,
            "save_total_limit": training_args.save_total_limit,
            "fp16": training_args.fp16,
            "seed": training_args.seed,
        }
        training_state_dict = {
            "global_step": state.global_step,
            "epoch": state.epoch,
            "best_metric": state.best_metric,
            "best_model_checkpoint": state.best_model_checkpoint,
        }
        return {
            "train_config": train_cfg_dict,
            "training_arguments": training_args_dict,
            "training_state": training_state_dict,
        }
class ClearCacheCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

class GradualUnfreezeCallback(TrainerCallback):
    """Unfreezes one encoder layer every `unfreeze_steps` steps, starting from the last layer."""
    def __init__(self, model, steps_between_unfreezes=1000, max_unfreeze_layers=3):
        self.model = model
        self.steps_between_unfreezes = steps_between_unfreezes
        self.max_unfreeze_layers = max_unfreeze_layers
        self.unfreeze_count = 0
        self.next_unfreeze_step = steps_between_unfreezes  # first unfreeze after this many steps
        
    def get_encoder_layers(self, model):
        """Return list of encoder layers (the transformer blocks)."""
        # Unwrap model if needed (e.g., Accelerator, DDP)
        model = self._unwrap_model(model)
        # RoBERTa / BERT / GraphCodeBERT
        if hasattr(model, 'roberta'):
            return model.roberta.encoder.layer
        elif hasattr(model, 'bert'):
            return model.bert.encoder.layer
        else:
            raise AttributeError("Cannot find encoder layers in model")
    
    def _unwrap_model(self, model):
        try:
            # If model is wrapped by Hugging Face's `Accelerator` or `DDP`
            return model.module if hasattr(model, 'module') else model
        except:
            return model
    
    def unfreeze_next_layer(self):
        """Unfreeze the next encoder layer (from last to first) if limit not reached."""
        if self.unfreeze_count >= self.max_unfreeze_layers:
            return
        layers = self.get_encoder_layers(self.model)
        total_layers = len(layers)
        # Index of the next layer to unfreeze (starting from last layer: -1, -2, ...)
        index = total_layers - 1 - self.unfreeze_count
        if index < 0:
            print(f"⚠️ Already unfrozen all {total_layers} layers, stopping.")
            return
        layer = layers[index]
        # Count frozen parameters before
        frozen_before = sum(not p.requires_grad for p in layer.parameters())
        # Unfreeze all parameters of this layer
        for param in layer.parameters():
            param.requires_grad = True
        # Count after
        frozen_after = sum(not p.requires_grad for p in layer.parameters())
        self.unfreeze_count += 1
        print(f"\n🔓 Gradual Unfreeze Step {self.unfreeze_count}: Unfroze encoder layer {index} (0-index from top). "
              f"Frozen params before: {frozen_before}, after: {frozen_after}")
        print(f"   Total trainable layers now: classifier + {self.unfreeze_count} encoder layer(s)")
    
    def on_step_end(self, args, state, control, **kwargs):
        if not self.model.training:
            return
        # Check if we've reached the next unfreezing step
        if state.global_step >= self.next_unfreeze_step and self.unfreeze_count < self.max_unfreeze_layers:
            self.unfreeze_next_layer()
            # Schedule the next unfreeze
            self.next_unfreeze_step += self.steps_between_unfreezes
            # Optionally log to wandb
            if args.report_to and "wandb" in args.report_to:
                try:
                    import wandb
                    wandb.log({"gradual_unfreeze/layers_unfrozen": self.unfreeze_count}, step=state.global_step)
                except:
                    pass
# ----------------------------------------------------------------------
# TrainConfig - central configuration object
# ----------------------------------------------------------------------
@dataclass
class TrainConfig:
    TEST: bool = False
    model_name: str = "microsoft/graphcodebert-base"
    output_dir: str = "./graphcodebert"
    num_epochs: int = 3
    max_steps: int = -1
    batch_size: int = 16
    learning_rate: float = 2e-5
    max_length: int = 512
    num_labels: int = 2
    use_wandb: bool = False
    freeze_base: bool = False

    # Loss types (ce, focal, infonce) – r-drop is separate flag
    loss_type: str = "ce"
    focal_alpha: float = 1.0
    focal_gamma: float = 2.0
    r_drop_alpha: float = 4.0
    infonce_temperature: float = 0.07
    infonce_weight: float = 0.5

    seed: int = 42
    resume_from_checkpoint: str = None

    save_steps: int = 100
    eval_steps: int = 100
    logging_steps: int = 5

    label_smoothing: float = 0.1
    adversarial_epsilon: float = 0.5
    use_swa: bool = True
    swa_start_epoch: int = 2
    swa_lr: float = 1e-5
    data_augmentation: bool = True
    aug_rename_prob: float = 0.3
    aug_format_prob: float = 0.3
    weight_decay: float = 0.1

    # MixCode and frequency consistency
    mixup_alpha: float = 1.0
    low_pass_keep_ratio: float = 0.5
    freq_consistency_weight: float = 0.1

    # Explicit toggles
    use_mixcode: bool = True
    use_fgm: bool = True
    fgm_freq: int = 1
    use_r_drop: bool = True
    use_freq_consistency_loss: bool = True

    # Attention spectral loss parameters
    use_attn_spectral: bool = True
    attn_spectral_weight: float = 0.1
    attn_spectral_cutoff_ratio: float = 0.25

    # Dropout
    hidden_dropout_prob: float = 0.3
    attention_probs_dropout_prob: float = 0.3
    classifier_dropout: float = 0.3

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch_compile: bool = True
    cache_dir: str = "./tokenized_cache"

    # Additional flags that might be used inside trainer
    use_swa_actual: bool = True          # alias for use_swa, kept for clarity
    use_fgm_actual: bool = True          # alias for use_fgm
    use_r_drop_actual: bool = True       # alias for use_r_drop
    use_mixcode_actual: bool = True      # alias for use_mixcode
    use_attn_spectral_actual: bool = True
    use_freq_consistency_loss_actual: bool = True

    # Memory / performance
    gradient_checkpointing: bool = True          # saves memory
    attn_downsample_size: int = 64               # max spatial size for attention loss
    attn_layers_to_keep: int = 2                 # use only last N layers for attention loss (-1 = all)
    
    # Fourier Mixup
    fourier_mixup_alpha: float = 0.5             # mixing strength (0=off)
    fourier_mixup_prob: float = 0.5              # probability per batch

    # Spectral normalisation
    use_spectral_norm: bool = False              # apply to classifier head

    # Gradual unfreezing
    gradual_unfreeze: bool = False           # Enable progressive unfreezing
    gradual_unfreeze_steps: int = 1000       # Unfreeze one layer every N steps
    gradual_unfreeze_max_layers: int = 3     # Maximum layers to unfreeze (excluding classifier)


    def __post_init__(self):
        # Ensure consistency: the trainer will use these aliases
        self.use_swa_actual = self.use_swa
        self.use_fgm_actual = self.use_fgm
        self.use_r_drop_actual = self.use_r_drop
        self.use_mixcode_actual = self.use_mixcode
        self.use_attn_spectral_actual = self.use_attn_spectral
        self.use_freq_consistency_loss_actual = self.use_freq_consistency_loss

    def copy(self) -> "TrainConfig":
        return TrainConfig(**asdict(self))

def validate_config(cfg: TrainConfig):
    """Check for conflicting/unsupported configuration options."""
    errors = []

    # Attention spectral loss requires output_attentions=True
    if cfg.use_attn_spectral and not cfg.use_attn_spectral:
        # Actually it's controlled inside load_model_and_tokenizer via cfg.use_attn_spectral
        # But we must ensure the model is loaded with output_attentions=True
        if not cfg.use_attn_spectral:  # placeholder – actual flag is same
            errors.append("use_attn_spectral=True but model not configured to output attentions")
    # More meaningful: Check that when use_attn_spectral is True, we will set output_attentions=True
    # Already done in load_model_and_tokenizer, but we can warn if disabled by user mistake.

    # FGM + MixCode: both modify embeddings, may interact unexpectedly
    if cfg.use_fgm and cfg.use_mixcode:
        print("⚠️ Warning: FGM and MixCode both modify embeddings. FGM will operate on the mixed embeddings. This is allowed but may behave differently from standard FGM.")

    # Label smoothing with focal loss: focal loss already has its own smoothing
    if cfg.loss_type == "focal" and cfg.label_smoothing > 0:
        errors.append("label_smoothing > 0 is not supported with focal loss (use focal's internal smoothing).")

    # InfoNCE requires hidden states
    if cfg.loss_type == "infonce" and not cfg.use_attn_spectral:  # just an example
        # In load_model_and_tokenizer, we set output_hidden_states=True for infonce, so fine.
        pass

    # TPU compatibility: if device is xla, certain features may not work (e.g., torch.compile)
    if "xla" in str(cfg.device) or (hasattr(torch, 'xla') and torch.xla.is_available()):
        if cfg.torch_compile:
            errors.append("torch_compile=True not compatible with TPU (XLA). Set torch_compile=False.")
        if cfg.use_swa:
            print("⚠️ SWA on TPU is experimental. May work but not fully tested.")

    if errors:
        raise ValueError("Configuration errors:\n" + "\n".join(f" - {e}" for e in errors))

    # Log warnings
    if cfg.max_length > 512:
        print("⚠️ max_length > 512 may cause OOM or very slow attention spectral loss.")


# ----------------------------------------------------------------------
# RobustTrainer - now accepts a single config object
# ----------------------------------------------------------------------
class RobustTrainer(Trainer):
    def __init__(
        self,
        *args,
        cfg: TrainConfig,                     # single config object
        processing_class,
        compute_loss_fn: Optional[Callable] = None,
        **kwargs
    ):
        # The base Trainer still needs its arguments (model, train_dataset, etc.)
        # We pass them via *args and **kwargs. The config is stored separately.
        super().__init__(*args, **kwargs)
        self.cfg = cfg
        self.processing_class = processing_class
        self.compute_loss_fn = compute_loss_fn

        # SWA state
        self.swa_model = None
        self.swa_scheduler = None
        self._swa_started = False

    def _unwrap_model(self, model):
        try:
            return self.accelerator.unwrap_model(model)
        except (AttributeError, TypeError):
            while hasattr(model, 'module'):
                model = model.module
            return model

    def attention_spectral_loss(self, attentions, labels):
        """Memory‑efficient version with downsampling and layer selection."""
        cfg = self.cfg
        
        # If attentions is tuple of layers, select only the last N layers
        if cfg.attn_layers_to_keep > 0 and isinstance(attentions, (tuple, list)):
            attentions = attentions[-cfg.attn_layers_to_keep:]
        
        total_loss = 0.0
        for layer_attn in attentions:   # layer_attn shape: [B, H, T, T]
            # Average over heads -> [B, T, T]
            attn_map = layer_attn.mean(dim=1)
            
            # Dynamic downsampling: if T > cfg.attn_downsample_size, downsample
            T = attn_map.size(-1)
            if T > cfg.attn_downsample_size:
                attn_map = F.adaptive_avg_pool2d(attn_map, 
                                                 (cfg.attn_downsample_size, cfg.attn_downsample_size))
                T = cfg.attn_downsample_size
            
            # 2D FFT
            fft = torch.fft.fft2(attn_map, dim=(-2, -1), norm='ortho')
            power = (fft.abs() ** 2)
            
            # Radial mask
            center = T // 2
            y, x = torch.meshgrid(torch.arange(T, device=power.device),
                                  torch.arange(T, device=power.device),
                                  indexing="ij")
            dist = torch.sqrt((x - center)**2 + (y - center)**2)
            max_radius = T * cfg.attn_spectral_cutoff_ratio
            low_mask = dist <= max_radius
            high_mask = dist > max_radius
            
            low_energy = power[:, low_mask].sum(dim=1)
            high_energy = power[:, high_mask].sum(dim=1)
            total_energy = low_energy + high_energy + 1e-8
            ratio = low_energy / total_energy
            targets = 1.0 - labels.float()
            layer_loss = F.binary_cross_entropy(ratio, targets)
            total_loss = total_loss + layer_loss
        
        # Average over selected layers
        total_loss = total_loss / len(attentions) if isinstance(attentions, (tuple, list)) else total_loss
        return total_loss
        
    def compute_loss(self, model, inputs, num_items_in_batch=None, return_outputs=False):
        cfg = self.cfg
        labels = inputs.get("labels", None)
        device = self.args.device
        unwrapped_model = self._unwrap_model(model)

        # MixCode (embedding mixup)
        use_mixcode_actual = cfg.use_mixcode and cfg.mixup_alpha > 0
        if use_mixcode_actual and model.training and labels is not None:
            batch_size = inputs['input_ids'].size(0)
            shuffle_idx = torch.randperm(batch_size, device=device)
            lam = np.random.beta(cfg.mixup_alpha, cfg.mixup_alpha)

            if hasattr(unwrapped_model, 'roberta'):
                emb_layer = unwrapped_model.roberta.embeddings.word_embeddings
            elif hasattr(unwrapped_model, 'bert'):
                emb_layer = unwrapped_model.bert.embeddings.word_embeddings
            else:
                raise AttributeError("Model must have 'roberta' or 'bert' for MixCode.")

            orig_embeds = emb_layer(inputs['input_ids'])
            shuffled_embeds = emb_layer(inputs['input_ids'][shuffle_idx])
            mixed_embeds = lam * orig_embeds + (1 - lam) * shuffled_embeds

            n_classes = unwrapped_model.config.num_labels
            one_hot = F.one_hot(labels, n_classes).float()
            one_hot_shuffled = F.one_hot(labels[shuffle_idx], n_classes).float()
            soft_labels = lam * one_hot + (1 - lam) * one_hot_shuffled

            forward_kwargs = {"inputs_embeds": mixed_embeds, "attention_mask": inputs['attention_mask']}
            is_mixup = True
        else:
            forward_kwargs = {k: v for k, v in inputs.items() if k != "labels"}
            soft_labels = None
            is_mixup = False

        # Forward pass
        outputs = model(**forward_kwargs)
        logits = outputs.logits

        if labels is None:
            raise ValueError("Labels are required for supervised loss computation.")

        # 1. Base Cross Entropy loss (with optional label smoothing)
        if cfg.label_smoothing > 0:
            n_classes = logits.size(-1)
            if is_mixup and soft_labels is not None:
                smooth_targets = soft_labels
            else:
                smooth_targets = torch.zeros_like(logits).fill_(cfg.label_smoothing / (n_classes - 1))
                smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - cfg.label_smoothing)
            base_ce = -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()
        else:
            if is_mixup and soft_labels is not None:
                log_probs = F.log_softmax(logits, dim=-1)
                base_ce = -(soft_labels * log_probs).sum(dim=-1).mean()
            else:
                base_ce = F.cross_entropy(logits, labels)

        total_loss = base_ce

        # 2. R-Drop regularisation
        if cfg.use_r_drop and cfg.r_drop_alpha > 0 and model.training and labels is not None:
            out2 = model(**forward_kwargs)
            logits2 = out2.logits

            p = F.log_softmax(logits, dim=-1)
            q = F.log_softmax(logits2, dim=-1)
            kl = (F.kl_div(p, F.softmax(logits2, dim=-1), reduction="batchmean") +
                  F.kl_div(q, F.softmax(logits, dim=-1), reduction="batchmean")) / 2

            if cfg.label_smoothing > 0:
                if is_mixup and soft_labels is not None:
                    smooth_targets = soft_labels
                else:
                    n_classes = logits.size(-1)
                    smooth_targets = torch.zeros_like(logits).fill_(cfg.label_smoothing / (n_classes - 1))
                    smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - cfg.label_smoothing)
                ce1 = -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()
                ce2 = -(smooth_targets * F.log_softmax(logits2, dim=-1)).sum(dim=-1).mean()
            else:
                if is_mixup and soft_labels is not None:
                    log_probs1 = F.log_softmax(logits, dim=-1)
                    log_probs2 = F.log_softmax(logits2, dim=-1)
                    ce1 = -(soft_labels * log_probs1).sum(dim=-1).mean()
                    ce2 = -(soft_labels * log_probs2).sum(dim=-1).mean()
                else:
                    ce1 = F.cross_entropy(logits, labels)
                    ce2 = F.cross_entropy(logits2, labels)

            rdrop_loss = (ce1 + ce2) / 2 + cfg.r_drop_alpha * kl
            total_loss = total_loss + (rdrop_loss - base_ce)

        # 3. Attention Spectral Loss
        if (cfg.use_attn_spectral and cfg.attn_spectral_weight > 0 and
            model.training and labels is not None and hasattr(outputs, 'attentions') and outputs.attentions is not None):
            attn_loss = self.attention_spectral_loss(outputs.attentions, labels)
            total_loss = total_loss + cfg.attn_spectral_weight * attn_loss

        # 4. Frequency-domain consistency loss (FFT low-pass)
        if cfg.use_freq_consistency_loss and cfg.freq_consistency_weight > 0 and model.training and labels is not None:
            # Assumes RoBERTa-like model; adjust if needed
            emb_layer = model.roberta.embeddings.word_embeddings
            orig_embeds = emb_layer(inputs['input_ids'])
            filtered_embeds = low_pass_filter_embeddings(orig_embeds, cfg.low_pass_keep_ratio)
            filtered_outputs = model(inputs_embeds=filtered_embeds, attention_mask=inputs['attention_mask'])
            kl = F.kl_div(
                F.log_softmax(outputs.logits, dim=-1),
                F.softmax(filtered_outputs.logits, dim=-1),
                reduction='batchmean'
            )
            total_loss = total_loss + cfg.freq_consistency_weight * kl

        return (total_loss, outputs) if return_outputs else total_loss

    def _fgm_attack(self, model, inputs):
        cfg = self.cfg
        unwrapped = self._unwrap_model(model)
        if hasattr(unwrapped, 'roberta'):
            emb_layer = unwrapped.roberta.embeddings.word_embeddings
        elif hasattr(unwrapped, 'bert'):
            emb_layer = unwrapped.bert.embeddings.word_embeddings
        else:
            return

        input_ids = inputs['input_ids']
        original_embeds = emb_layer(input_ids).detach().clone()
        original_embeds.requires_grad = True

        forward_kwargs = {
            "inputs_embeds": original_embeds,
            "attention_mask": inputs['attention_mask']
        }
        outputs = model(**forward_kwargs)
        loss = F.cross_entropy(outputs.logits, inputs['labels'])
        grads = torch.autograd.grad(loss, original_embeds, retain_graph=False)[0]

        if grads is not None:
            norm = grads.norm()
            if norm > 1e-8 and not torch.isnan(norm):
                perturbation = cfg.adversarial_epsilon * (grads / norm)
                perturbed_embeds = original_embeds + perturbation
                adv_outputs = model(
                    inputs_embeds=perturbed_embeds,
                    attention_mask=inputs['attention_mask']
                )
                adv_loss = F.cross_entropy(adv_outputs.logits, inputs['labels'])
                self.accelerator.backward(adv_loss)

    def training_step(self, model, inputs, num_items_in_batch=None):
        cfg = self.cfg
        model.train()
        inputs = self._prepare_inputs(inputs)

        # Original forward & backward (without FGM yet)
        loss, outputs = self.compute_loss(model, inputs, return_outputs=True)
        if self.args.n_gpu > 1:
            loss = loss.mean()
        self.accelerator.backward(loss)

        # FGM adversarial training (on embeddings)
        if cfg.use_fgm and cfg.adversarial_epsilon > 0 and model.training:
            if self.state.global_step % cfg.fgm_freq == 0:
                self._fgm_attack(model, inputs)

        # SWA update
        if cfg.use_swa:
            if not self._swa_started and self.state.epoch >= cfg.swa_start_epoch:
                self._swa_started = True
                self.swa_model = AveragedModel(model)
                self.swa_scheduler = SWALR(self.optimizer, swa_lr=cfg.swa_lr)
                self.log({"SWA": "started"})
            if self._swa_started and self.swa_model is not None:
                self.swa_model.update_parameters(model)

        return loss

    def optimizer_step(self, model, optimizer, optimizer_idx=None):
        super().optimizer_step(model, optimizer, optimizer_idx)
        if self._swa_started and self.swa_scheduler is not None:
            self.swa_scheduler.step()

    def save_model(self, output_dir: Optional[str] = None, _internal_call: bool = False):
        if self._swa_started and self.swa_model is not None:
            if self.train_dataset is not None:
                train_loader = self.get_train_dataloader()
                update_bn(train_loader, self.swa_model)
            real_model = self.swa_model.module if hasattr(self.swa_model, "module") else self.swa_model
            real_model = self._unwrap_model(real_model)
            real_model.save_pretrained(output_dir or self.args.output_dir)
        else:
            super().save_model(output_dir, _internal_call)
    
        save_dir = output_dir or self.args.output_dir
    
        if self.processing_class is not None:
            self.processing_class.save_pretrained(save_dir)
            print(f"Tokenizer saved to {save_dir}")
    
        config_dict = asdict(self.cfg)
    
        def serialize(obj):
            if isinstance(obj, Path):
                return str(obj)
            if isinstance(obj, torch.device):
                return str(obj)
            if hasattr(obj, "value"):
                return obj.value
            return obj
    
        safe_dict = {k: serialize(v) for k, v in config_dict.items()}
    
        config_path = os.path.join(save_dir, "train_config.yaml")
        with open(config_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(safe_dict, f, sort_keys=False, allow_unicode=True)
    
        print(f"TrainConfig saved to {config_path}")

# ----------------------------------------------------------------------
# Updated train_pipeline - passes the full config to RobustTrainer
# ----------------------------------------------------------------------
def train_pipeline(cfg: TrainConfig):
    os.makedirs(cfg.output_dir, exist_ok=True)
    logger = setup_logger(cfg.output_dir, "training")
    set_seed(cfg.seed)
    validate_config(cfg)
    log_training_config(cfg, logger)

    model, tokenizer = load_model_and_tokenizer(cfg)
    # ----- Freezing strategy -----
    if cfg.gradual_unfreeze:
        # Freeze everything except classifier head (the base model fully frozen)
        for name, param in model.named_parameters():
            if "classifier" not in name and "cls" not in name.lower():
                param.requires_grad = False
        logger.info("Initial freezing: only classifier head trainable (for gradual unfreezing).")
        # We'll later unfreeze layers via callback
    elif cfg.freeze_base:
        freeze_base_model(model)
        logger.info("Base model frozen (traditional full freeze).")
    else:
        logger.info("No freezing applied – all parameters trainable.")

    # Device handling
    if "xla" in str(cfg.device) or (hasattr(torch, 'xla') and xm.xrt_world_size() > 1):
        device = xm.xla_device()
        logger.info("Using TPU device")
    else:
        device = cfg.device
    model.to(device)
    logger.info(f"Model placed on {device}")

    if cfg.freeze_base:
        freeze_base_model(model)

    log_model_architecture(model, tokenizer, logger)

    if cfg.loss_type == "infonce":
        model.config.output_hidden_states = True
        logger.info("Enabled hidden states for InfoNCE loss.")

    # Data augmentation
    aug = None
    if cfg.data_augmentation:
        aug = CodeAugmentation(rename_prob=cfg.aug_rename_prob,
                               format_prob=cfg.aug_format_prob,
                               mixup_alpha=cfg.mixup_alpha)
        logger.info(f"Data augmentation enabled (rename={cfg.aug_rename_prob}, format={cfg.aug_format_prob})")

    train_dataset, val_dataset = load_datasets(
        tokenizer, cfg.max_length, aug=aug, limit=100 if cfg.TEST else None, cache_dir=cfg.cache_dir
    )

    if cfg.use_wandb:
        try:
            wandb_dir = configure_wandb_offline(cfg.output_dir)
            logger.info(f"W&B offline logging enabled. Local logs: {wandb_dir}")
        except Exception as e:
            logger.warning(f"W&B import failed ({e}); proceeding without it.")
            cfg.use_wandb = False

    steps_per_epoch = max(1, len(train_dataset) // cfg.batch_size)
    total_steps = cfg.num_epochs * steps_per_epoch
    warmup_steps = max(1, int(total_steps * 0.1))
    torch.backends.cudnn.benchmark = True

    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.num_epochs,
        max_steps=cfg.max_steps,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size * 2,
        learning_rate=cfg.learning_rate,
        warmup_steps=warmup_steps,
        weight_decay=cfg.weight_decay,
        logging_strategy="steps",
        logging_steps=cfg.logging_steps,
        eval_strategy="steps",
        eval_steps=cfg.eval_steps,
        save_strategy="steps",
        save_steps=cfg.save_steps,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=5,
        report_to=["wandb"] if cfg.use_wandb else [],
        run_name=f"graphcodebert-lr={cfg.learning_rate}-batchsize={cfg.batch_size}" if cfg.use_wandb else None,
        dataloader_pin_memory=torch.cuda.is_available(),
        seed=cfg.seed,
        torch_compile=cfg.torch_compile,
        torch_compile_mode="reduce-overhead",
        fp16=False,
        bf16=True,
        lr_scheduler_type="cosine_with_restarts",
        lr_scheduler_kwargs={"num_cycles": 2},
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    # Build loss function if needed
    compute_loss_fn = None
    if cfg.loss_type == "focal":
        compute_loss_fn = get_focal_loss(cfg.focal_alpha, cfg.focal_gamma, smoothing=cfg.label_smoothing)
    elif cfg.loss_type == "infonce":
        compute_loss_fn = get_infonce_loss(cfg.infonce_temperature, cfg.infonce_weight)

    callbacks = [
        EarlyStoppingCallback(early_stopping_patience=3),
        ConfigLoggingCallback(train_config=cfg),
        ClearCacheCallback()
    ]
    
    if cfg.gradual_unfreeze:
        # We must pass the model (unwrapped) to the callback – but the model may be wrapped later by Accelerator.
        # The callback will unwrap itself on each call.
        callbacks.append(GradualUnfreezeCallback(
            model=model,
            steps_between_unfreezes=cfg.gradual_unfreeze_steps,
            max_unfreeze_layers=cfg.gradual_unfreeze_max_layers
        ))
        logger.info(f"Added GradualUnfreezeCallback: unfreeze every {cfg.gradual_unfreeze_steps} steps, max {cfg.gradual_unfreeze_max_layers} layers")
    
    trainer = RobustTrainer(
        model=model,
        cfg=cfg,
        processing_class=tokenizer,
        compute_loss_fn=compute_loss_fn,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        callbacks=callbacks,
    )

    logger.info("=== Starting training ===")
    try:
        trainer.train(resume_from_checkpoint=cfg.resume_from_checkpoint)
        logger.info("Training completed successfully.")
    except KeyboardInterrupt:
        logger.info("Training interrupted by user.")
        raise
    except Exception as e:
        logger.error(f"Training failed: {e}")
        raise
    finally:
        if cfg.use_wandb:
            wandb.finish()

    final_save_dir = os.path.join(cfg.output_dir, "final_model")
    trainer.save_model(final_save_dir)
    logger.info(f"Final model and tokenizer saved to {final_save_dir}")
    return trainer, trainer.model, tokenizer

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ----------------------------------------------------------------------
# Main execution
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # Set matmul precision for speed/stability
    torch.set_float32_matmul_precision('medium')

    # Cleanup
    wandb.finish()
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    cfg = TrainConfig(
        # Model and Data paths
        model_name="/kaggle/input/models/dzung271828/microsoft-graphcodebert-base/transformers/default/1",
        output_dir="training/rdrop-spectralnorm-attentionmaps/",
        
        # Training Hyperparameters
        num_epochs=5,
        batch_size=128,
        learning_rate=1e-6,
        weight_decay=0.1,
        max_length=512,
        torch_compile=True,
        
        gradual_unfreeze=False,
        gradual_unfreeze_steps=2000,
        gradual_unfreeze_max_layers=3,
        freeze_base=True,
        
        # Logging and Checkpointing
        save_steps=500,
        eval_steps=500,
        logging_steps=10,
        use_wandb=False,
        
        # Loss Configuration
        # Note: R-Dop uses standard CE internally, but 'r-drop' indicates the wrapper usage
        loss_type="ce",           
        r_drop_alpha=5.0,         
        label_smoothing=0.3,
        
        # SWA (Stochastic Weight Averaging)
        use_swa=False,
        swa_start_epoch=2,
        swa_lr=5e-6,
        
        # Data Augmentation
        data_augmentation=True,
        aug_rename_prob=0.5,
        aug_format_prob=0.5,
        
        # Adversarial Training (FGM)
        use_fgm=False,
        fgm_freq=5,
        adversarial_epsilon=0.5,
        
        # MixCode
        use_mixcode=False,
        mixup_alpha=1.0,
        
        # Regularization (Dropout)
        hidden_dropout_prob=0.3,
        attention_probs_dropout_prob=0.3,
        classifier_dropout=0.5,
        
        # Attention Spectral Loss
        use_attn_spectral=True,      
        attn_spectral_weight=0.1,
        attn_spectral_cutoff_ratio=0.25,

        # Fourier transform
        use_freq_consistency_loss=True,
        freq_consistency_weight=0.3,

        # Spectral normalisation
        use_spectral_norm=False,

        
        # Flags
        TEST=False,
        resume_from_checkpoint=None,
    )

    trainer, model, tokenizer = train_pipeline(cfg)

In [ ]:
!rm -rf tokenized_cache

In [ ]:
import os
import shutil
from IPython.display import FileLink, display

base_dir = "graphcodebert-swa-from-epoch-1"
zip_paths = []

for name in os.listdir(base_dir):
    full_path = os.path.join(base_dir, name)
    
    if name.startswith("checkpoint-") and os.path.isdir(full_path):
        zip_name = f"{name}.zip"
        
        shutil.make_archive(name, 'zip', full_path)
        zip_paths.append(zip_name)

for zp in zip_paths:
    display(FileLink(zp))

In [ ]:
import os
import json
import logging
import sys
from pathlib import Path
from typing import Dict, Optional
import numpy as np
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ---------- copy the exact TrainConfig dataclass from training ----------
from dataclasses import dataclass, asdict, field
from typing import Optional, Callable
# ---------- Helper to load config from checkpoint ----------
def load_training_config_from_checkpoint(checkpoint_path: str) -> TrainConfig:
    """Load the TrainConfig used during training from JSON."""
    # Look for train_config.json (saved by trainer.save_model)
    config_path = Path(checkpoint_path) / "train_config.json"
    if not config_path.exists():
        # Fallback: config_hyperparams.json saved by ConfigLoggingCallback in checkpoints
        config_path = Path(checkpoint_path) / "config_hyperparams.json"
    if not config_path.exists():
        raise FileNotFoundError(f"No training config found in {checkpoint_path}")
    with open(config_path, "r") as f:
        config_dict = json.load(f)
    # The JSON may contain nested 'train_config' field (from ConfigLoggingCallback)
    if "train_config" in config_dict:
        config_dict = config_dict["train_config"]
    # Convert types (e.g., device string to torch.device)
    if "device" in config_dict and isinstance(config_dict["device"], str):
        config_dict["device"] = torch.device(config_dict["device"])
    # Recreate TrainConfig
    return TrainConfig(**config_dict)

# ---------- Model loader that respects training architecture ----------
def load_model_and_tokenizer_from_checkpoint(checkpoint_path: str, cfg: TrainConfig):
    """Load model and tokenizer with the same architectural modifications as training."""
    # Tokenizer (fine-tuned)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    # Base config from checkpoint (preserves original vocab size etc.)
    config = AutoConfig.from_pretrained(checkpoint_path)
    # Override with training config values
    config.num_labels = cfg.num_labels
    config.problem_type = "single_label_classification"
    config.hidden_dropout_prob = cfg.hidden_dropout_prob
    config.attention_probs_dropout_prob = cfg.attention_probs_dropout_prob
    if hasattr(config, 'classifier_dropout'):
        config.classifier_dropout = cfg.classifier_dropout
    # Enable attention outputs if spectral loss was used
    if cfg.use_attn_spectral:
        config.output_attentions = True
    # Load model with the modified config
    model = AutoModelForSequenceClassification.from_config(config)
    
    # Apply spectral normalisation to classifier.dense if used during training
    if cfg.use_spectral_norm:
        if hasattr(model, 'classifier') and hasattr(model.classifier, 'dense'):
            model.classifier.dense = torch.nn.utils.parametrizations.spectral_norm(
                model.classifier.dense
            )
            print("Applied spectral norm to classifier.dense (inference)")
        else:
            print("Warning: spectral norm requested but classifier.dense not found")
    from safetensors.torch import load_file
    state_dict = load_file(checkpoint_path + "/model.safetensors")
    model.load_state_dict(state_dict, strict=True)
    return model, tokenizer

# ---------- Logger setup (same as training) ----------
def setup_logger(output_dir: str, name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(console_handler)
    os.makedirs(output_dir, exist_ok=True)
    log_path = Path(output_dir) / f"{name}.log"
    file_handler = logging.FileHandler(log_path, mode="w")
    file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(file_handler)
    return logger

def log_model_architecture_inference(model, tokenizer, logger):
    logger.info("===== Model Architecture =====")
    logger.info("\n" + model.__repr__())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params
    logger.info("===== Parameter Summary =====")
    logger.info(f"Total Parameters:         {total_params:,}")
    logger.info(f"Trainable Parameters:     {trainable_params:,}")
    logger.info(f"Non-trainable Parameters: {non_trainable_params:,}")
    logger.info("===== Tokenizer Summary =====")
    logger.info(f"Vocab size: {len(tokenizer)} | Special tokens: {tokenizer.all_special_tokens}")
    logger.info("===== End of Architecture Log =====")

# ---------- Main inference function (rewritten) ----------
def run_standalone_inference(
    checkpoint_path: str,
    output_dir: str = "./",
    output_csv: str = "submission.csv",
    batch_size: int = 32,
    max_length: int = 512,
    dataset_name: str = "DaniilOr/SemEval-2026-Task13",
    dataset_config: str = "A",
    split: str = "test",
):
    # Create output directory
    checkpoint_name = Path(checkpoint_path).name
    save_dir = Path(output_dir) / checkpoint_name
    save_dir.mkdir(parents=True, exist_ok=True)
    logger = setup_logger(save_dir, "inference")

    # ---- 1. Load training config to know architecture flags ----
    logger.info(f"Loading training configuration from {checkpoint_path}")
    cfg = load_training_config_from_checkpoint(checkpoint_path)
    # Override max_length if needed (but keep training config's value for consistency)
    cfg.max_length = max_length
    logger.info(f"Loaded config: use_attn_spectral={cfg.use_attn_spectral}, use_spectral_norm={cfg.use_spectral_norm}")

    # ---- 2. Load model and tokenizer with exact training architecture ----
    logger.info(f"Loading model and tokenizer from {checkpoint_path}")
    model, tokenizer = load_model_and_tokenizer_from_checkpoint(checkpoint_path, cfg)

    # ---- 3. Device and evaluation mode ----
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    log_model_architecture_inference(model, tokenizer, logger)

    # ---- 4. Load dataset (memory efficient) ----
    logger.info(f"Loading dataset from: {dataset_name}")
    if ".parquet" in dataset_name.lower():
        logger.info("Detected .parquet file – loading with datasets (memory-mapped)")
        test_ds = load_dataset("parquet", data_files=dataset_name, split="train")
        logger.info(f"Loaded Parquet file with {len(test_ds)} examples")
        if split != "test":
            logger.warning(f"Ignoring 'split={split}' – loading full Parquet file instead.")
    else:
        test_ds = load_dataset(dataset_name, dataset_config, split=split)

    # ---- 5. Tokenize ----
    def tokenize_fn(examples):
        return tokenizer(
            examples["code"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    logger.info("Tokenizing dataset")
    tokenized_ds = test_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=[c for c in test_ds.column_names if c not in ["input_ids", "attention_mask", "id", "label"]],
        desc="Tokenising",
        num_proc=os.cpu_count(),
    )
    tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    test_loader = torch.utils.data.DataLoader(
        tokenized_ds,
        batch_size=batch_size,
        shuffle=False,
    )

    # ---- 6. Inference ----
    logger.info(f"Running inference on {len(test_ds)} examples...")
    all_logits = []

    # Optional torch compilation (same as training)
    if cfg.torch_compile:
        model = torch.compile(model, mode="reduce-overhead")
        logger.info("Model compiled with torch.compile")

    with torch.inference_mode():
        for batch in tqdm(test_loader, desc="Predicting"):
            inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
            outputs = model(**inputs)
            all_logits.append(outputs.logits.cpu())

    logits = torch.cat(all_logits, dim=0).numpy()
    pred_labels = logits.argmax(axis=-1)

    # ---- 7. Metrics (if labels available) ----
    if "label" in test_ds.column_names:
        logger.info("Calculating classification metrics...")
        true_labels = np.array(test_ds["label"])
        acc = accuracy_score(true_labels, pred_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, pred_labels, average='weighted'
        )
        cm = confusion_matrix(true_labels, pred_labels)
        logger.info(f"Accuracy:  {acc:.4f}")
        logger.info(f"Precision: {precision:.4f}")
        logger.info(f"Recall:    {recall:.4f}")
        logger.info(f"F1-Score:  {f1:.4f}")
        logger.info(f"Confusion Matrix:\n{cm}")
        metrics = {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1, "confusion_matrix": cm.tolist()}
        with open(save_dir / "metrics.json", "w") as f:
            json.dump(metrics, f, indent=4)
    else:
        logger.warning("No 'label' column found. Skipping metrics.")

    # ---- 8. Save predictions CSV ----
    if "id" in test_ds.column_names:
        ids = test_ds["id"]
    elif "ID" in test_ds.column_names:
        ids = test_ds["ID"]
    else:
        ids = list(range(len(pred_labels)))

    csv_path = save_dir / output_csv
    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("ID,label\n")
        for idx, label in zip(ids, pred_labels):
            f.write(f"{idx},{label}\n")
    logger.info(f"✅ Predictions saved to {csv_path}")

# ---------- Example usage ----------
if __name__ == "__main__":
    # Optional: set precision flags
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('medium')

    CHECKPOINTS = ["training/rdrop-spectralnorm/checkpoint-500",
                "training/rdrop-spectralnorm/checkpoint-1000",
                "training/rdrop-spectralnorm/checkpoint-1500",
                "training/rdrop-spectralnorm/checkpoint-2000",
                "training/rdrop-spectralnorm/checkpoint-2500",
                  ]
    OUTPUT_DIR = "outputs"

    # Clean memory before inference
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for CHECKPOINT in CHECKPOINTS:
        ckpt_names = CHECKPOINT.split("/")
        num_ckpt = ckpt_names[-1].split("-")[-1]
        if os.path.exists(CHECKPOINT):
            run_standalone_inference(
                checkpoint_path=CHECKPOINT,
                output_dir=OUTPUT_DIR,
                output_csv=f"fourier-rdrop-spectralnorm-ckpt{num_ckpt}.csv",
                batch_size=500,
                dataset_name="/kaggle/input/datasets/nesfan/requirements/test.parquet",
                split="test"
            )

In [ ]:
import subprocess
subprocess.run(["cat", CHECKPOINT + "/train_config.yaml"], text=True)

In [ ]:
OUTPUT_CHECKPOINTS = "training/rdrop/checkpoint-3500"
DESTINATION = "-".join(OUTPUT_CHECKPOINTS.split("/"))
!zip -r {DESTINATION} {OUTPUT_CHECKPOINTS}

In [ ]:
!rm -rf outputs

In [ ]:
from IPython.display import FileLink
FileLink('spectral-attention.zip')

In [ ]:
import subprocess
subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')